# Stage 1: Per-Wallet Copy Sizing (tier3@2-0)

Fit per-wallet copy weights ``alpha_w`` on **train** per-wallet daily pnl
(3 tiers by Sharpe proxy: top 2x, middle 1x, bottom dropped), pick
hyperparameters on **validation** by sim Sharpe, single **test** pass.
Copy qty is capped by the reconstructed share-depth ``bucket_avail_copy_qty``.

**Output:** `stage1_scaled_result.json` + `signal_lab/wallet_scaling_{sim,ci,contrib}.csv`


In [ ]:
# Setup: imports, paths, constants
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

NB_DIR = Path.cwd() if "__file__" not in globals() else Path(__file__).resolve().parent
sys.path.insert(0, str(NB_DIR))
OUT_DIR = NB_DIR / "signal_lab"

import numpy as np
import pandas as pd

from lib import DEFAULT_TAGS
from signal_lab.filters import COPY_DEFAULT
from signal_lab.signal_lib import spearman_rho
from signal_lab.sizing import (
    block_bootstrap_sharpe,
    capital_constrained_sim,
    sizing_sharpe,
)
from signal_lab.stage1 import candidate_splits_for, load_stage1_data
from signal_lab.wallet_scaling import (
    alpha_kelly,
    alpha_tier,
    attach_depth_cap,
    run_sim,
    sim_row,
    wallet_daily_pnl,
    wallet_stats,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

BUDGET = 10_000.0
COST_SEL = 10.0
ALPHA_MAX_GRID = (2.0, 4.0, 8.0)
TIER_GRID = [(nt, am, amin) for nt in (3, 4, 5) for am in ALPHA_MAX_GRID for amin in (0.0, 0.25)]
UNIFORM_K_GRID = (0.5, 1.0, 2.0, 4.0)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [18]:
df_full, df_train, df_val, df_test, wallet_metrics, hold_metrics = load_stage1_data(tags=DEFAULT_TAGS)
print(f"df_full: {len(df_full):,}")
print(f"  train: {len(df_train):,}  val: {len(df_val):,}  test: {len(df_test):,}")


Markets: 2165960
Filtered markets for {'Geopolitics', 'Politics', 'Elections'}: 46780
Loading 16 trade shards...
Total trades loaded: 13,364,488
Unique wallets: 34,382
Date range: 2025-01-01 00:00:59+00:00 -> 2026-08-03 16:01:48+00:00
Chronological split: train <= 2025-09-15T00:00:00Z, val <= 2026-03-10T00:00:00Z, test > 2026-03-10T00:00:00Z
Method: chronological  |  Unique end dates: 520  (train=208, val=156, test=156)

  Train:  1,551,662 trades  (3,287 markets)
  Val:    4,273,763 trades  (6,163 markets)
  Test:   7,539,063 trades  (11,743 markets)
  Total: 13,364,488 trades  (21,193 markets)


/Users/vobornij/projects/polymarket/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


df_full: 13,364,488
  train: 1,551,662  val: 4,273,763  test: 7,539,063


## Copy universe

Candidate wallets = `COPY_DEFAULT` (copy-default filter).

In [19]:
wallets = set(COPY_DEFAULT(wallet_metrics, hold_metrics))
print(f"copy_default wallets: {len(wallets)}")


copy_default wallets: 45


## Share-depth cap

Cap = stage0 Phase 2's per-bucket max copy quantity (`avail_copy_qty`), exported with the processed trades.

In [20]:
splits = candidate_splits_for(df_full, wallets)
splits = attach_depth_cap(splits)
del df_full, df_train, df_val, df_test

for name in ("train", "val", "test"):
    fr = splits[name]
    capped = (fr["bucket_avail_copy_qty"] < fr["copyable_qty"]).mean()
    print(f"{name:5s}: {len(fr):,}  trades_capped_by_depth={capped:.3f}")


Chronological split: train <= 2025-09-19T00:00:00Z, val <= 2026-03-04T00:00:00Z, test > 2026-03-04T00:00:00Z
Method: chronological  |  Unique end dates: 457  (train=182, val=137, test=138)

  Train:     20,308 trades  (1,985 markets)
  Val:       21,757 trades  (2,510 markets)
  Test:       9,654 trades  (1,613 markets)
  Total:     51,719 trades  (6,108 markets)
train: 20,308  trades_capped_by_depth=0.000
val  : 21,757  trades_capped_by_depth=0.000
test : 9,654  trades_capped_by_depth=0.000


## Train per-wallet stats

Per-wallet daily pnl (copyable, alpha=1) with mean/std shrinkage -> Sharpe proxy.

In [21]:
train_daily = wallet_daily_pnl(splits["train"])
st = wallet_stats(train_daily)
print(f"wallets with train daily series: {len(st)}")
st[["mu", "sigma", "n_days", "sharpe_proxy", "total_pnl"]].sort_values(
    "sharpe_proxy", ascending=False
).head(15)


wallets with train daily series: 45


,mu,sigma,n_days,sharpe_proxy,total_pnl
wallet,,,,,
0x183a2a0b034877e761af0778da5134bebc37d514,3758.7952,10574.7140,10,0.2060,37587.9521
0xf72044fb3f98854411a069d9a682e54e4b824599,360.2479,1967.5725,38,0.1310,13689.4217
0xc5fe4e2927af431482ae4bb820b9e122c7c4e320,329.3853,781.7847,16,0.1159,5270.1650
0x89c73a6b97b2e58972fb83b7d661121814ba3ff0,266.6527,969.9679,22,0.1106,5866.3605
0x70c907411393dd590ef46908a80bb4cfb3dfcef3,218.4473,687.2109,27,0.1091,5898.0758
0x2625e0f62524fea18c26edd95877ce1710c8b2f8,246.6406,585.3277,16,0.0956,3946.2495
0xd3989ba133ab48b5b3a81e3dba9b37b5966a46d7,139.7143,1199.3943,106,0.0943,14809.7197
0x57c5de7efafb020e589f80e0da6e16e9b5c907aa,140.1978,723.7624,46,0.0937,6449.0966
0xf59fe344fb8af90a14bd6f2fe62430e56dc03be9,112.0526,592.5222,67,0.0926,7507.5252


## Weight schemes

All benchmarked vs copy-all: shrunk max-Sharpe (Kelly), tier, uniform-k.

In [22]:
schemes = {}
for am in ALPHA_MAX_GRID:
    schemes[f"kelly@{am:g}"] = ("kelly", alpha_kelly(st, am), {"alpha_max": am})
for (nt, am, amin) in TIER_GRID:
    schemes[f"tier{nt}@{am:g}-{amin:g}"] = (
        "tier",
        alpha_tier(st, nt, am, amin),
        {"n_tiers": nt, "alpha_max": am, "alpha_min": amin},
    )
for k in UNIFORM_K_GRID:
    schemes[f"uniform@{k:g}"] = ("uniform", pd.Series(k, index=st.index), {"k": k})
schemes["copy_all"] = ("copy_all", pd.Series(1.0, index=st.index), {})

print(f"schemes: {len(schemes)}")


schemes: 26


## Validation grid search

Objective: annualized Sharpe of daily resolution-pnl, fixed $10k budget, 10bps.

In [23]:
sim_rows = []
best_per_scheme = {}
for name, (scheme, alpha_map, params) in schemes.items():
    res = run_sim(splits["val"], alpha_map, COST_SEL)
    row = sim_row(scheme, name, "val", res)
    sim_rows.append(row)
    key = scheme if scheme != "kelly" else "kelly"
    if key not in best_per_scheme or row["sharpe_daily"] > best_per_scheme[key][2]:
        best_per_scheme[key] = (name, params, row["sharpe_daily"])

sim_df = pd.DataFrame(sim_rows)
sim_df[sim_df["split"] == "val"].sort_values("sharpe_daily", ascending=False).head(15)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
3,tier,tier3@2-0,val,3690,29786.9800,0.3822,1.7530,8081.1000,9999.9800
5,tier,tier3@4-0,val,3690,29786.9800,0.3822,1.7530,8081.1000,9999.9800
7,tier,tier3@8-0,val,3690,29786.9800,0.3822,1.7530,8081.1000,9999.9800
21,uniform,uniform@0.5,val,7882,19385.8600,0.2192,1.5150,7383.0400,10000.0000
19,tier,tier5@8-0,val,5472,27389.6200,0.3130,1.3960,8174.2900,9999.9900
17,tier,tier5@4-0,val,5472,27389.6200,0.3130,1.3960,8174.2900,9999.9900
15,tier,tier5@2-0,val,5472,27389.6200,0.3130,1.3960,8174.2900,9999.9900
13,tier,tier4@8-0,val,5482,26766.1800,0.3036,1.3770,8185.6700,10000.0000
11,tier,tier4@4-0,val,5482,26766.1800,0.3036,1.3770,8185.6700,10000.0000
9,tier,tier4@2-0,val,5482,26766.1800,0.3036,1.3770,8185.6700,10000.0000


In [24]:
print("Selected per scheme (by val Sharpe):")
for key, (name, params, val_sharpe) in best_per_scheme.items():
    print(f"  {key:10s} -> {name:>22s}  val_sharpe={val_sharpe:.3f}")

best_name = max(
    best_per_scheme.values(), key=lambda x: x[2]
)[0]
print(f"\nBest val config overall: {best_name}")


Selected per scheme (by val Sharpe):
  kelly      ->                kelly@2  val_sharpe=1.067
  tier       ->              tier3@2-0  val_sharpe=1.753
  uniform    ->            uniform@0.5  val_sharpe=1.515
  copy_all   ->               copy_all  val_sharpe=1.248

Best val config overall: tier3@2-0


## Test: single pass per chosen config

One honest test pass for each scheme's val-chosen config (10bps).

In [25]:
for key, (name, params, _val_sharpe) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, COST_SEL)
    row = sim_row(schemes[name][0], name, "test", res)
    sim_rows.append(row)

sim_df = pd.DataFrame(sim_rows)
sim_df.to_csv(OUT_DIR / "wallet_scaling_sim.csv", index=False)
sim_df[sim_df["split"] == "test"].sort_values("sharpe_daily", ascending=False)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
27,tier,tier3@2-0,test,650,31966.1800,0.7366,2.0290,1968.7100,10000.0000
28,uniform,uniform@0.5,test,4089,10965.4700,0.1656,0.5780,2243.7000,10000.0000
26,kelly,kelly@2,test,3775,5993.6300,0.0880,0.5660,2597.1400,10000.0000
29,copy_all,copy_all,test,3731,2171.7600,0.0289,0.2460,2776.5800,10000.0000


## Robustness: cost sweep + bootstrap CI

Cost sweep (0/10/30bps) + 7-day block-bootstrap Sharpe CI on test.

In [26]:
ci_rows = []
for key, (name, params, _) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    for cost in (0.0, 10.0, 30.0):
        res = run_sim(splits["test"], alpha_map, cost)
        point, lo, hi = block_bootstrap_sharpe(res["daily_pnl"], block_size=7, n_iter=1000, seed=42)
        ci_rows.append({
            "design": name, "cost_bps": cost,
            "pnl": round(res["net_pnl"], 2),
            "roi_w": round(res["net_pnl"] / res["notional"], 4) if res["notional"] > 0 else np.nan,
            "sharpe_daily": round(sizing_sharpe(res["daily_pnl"], 365.0), 3),
            "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
        })

res_all = capital_constrained_sim(splits["test"], "score1", BUDGET, 1.0, cost_bps=COST_SEL)
point, lo, hi = block_bootstrap_sharpe(res_all["daily_pnl"], block_size=7, n_iter=1000, seed=42)
ci_rows.append({
    "design": "copy_all", "cost_bps": COST_SEL,
    "pnl": round(res_all["net_pnl"], 2),
    "roi_w": round(res_all["net_pnl"] / res_all["notional"], 4) if res_all["notional"] > 0 else np.nan,
    "sharpe_daily": round(sizing_sharpe(res_all["daily_pnl"], 365.0), 3),
    "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
})

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUT_DIR / "wallet_scaling_ci.csv", index=False)
ci_df


,design,cost_bps,pnl,roi_w,sharpe_daily,ci_lo,ci_hi
0,kelly@2,0.0000,6061.7300,0.0890,0.5660,-1.3120,2.7130
1,kelly@2,10.0000,5993.6300,0.0880,0.5660,-1.3120,2.7130
2,kelly@2,30.0000,5857.4400,0.0860,0.5660,-1.3120,2.7130
3,tier3@2-0,0.0000,32009.5800,0.7376,2.0290,-2.1280,2.1280
4,tier3@2-0,10.0000,31966.1800,0.7366,2.0290,-2.1280,2.1280
5,tier3@2-0,30.0000,31879.3800,0.7346,2.0290,-2.1280,2.1280
6,uniform@0.5,0.0000,11031.6900,0.1666,0.5780,-1.8890,2.0150
7,uniform@0.5,10.0000,10965.4700,0.1656,0.5780,-1.8890,2.0150
8,uniform@0.5,30.0000,10833.0400,0.1636,0.5780,-1.8890,2.0150
9,copy_all,0.0000,2246.9100,0.0299,0.2460,-2.2960,1.7350


## Per-wallet contributions

Train alphas vs forward (test) wallet stats.

In [27]:
test_daily = wallet_daily_pnl(splits["test"])
test_st = test_daily.groupby("wallet")["copyable_pnl"].agg(
    test_pnl="sum", test_n_days="size"
)
test_sharpe = test_daily.groupby("wallet")["copyable_pnl"].apply(
    lambda s: (s.mean() / s.std() * np.sqrt(365.0)) if s.std() > 0 and len(s) >= 2 else np.nan
).rename("test_sharpe")

contrib = st.join(test_st, how="outer").join(test_sharpe, how="outer").fillna(0.0)
contrib = contrib[contrib["test_n_days"] > 0]
alpha_cont = schemes[best_per_scheme["kelly"][0]][1]
alpha_tier_cont = schemes[best_per_scheme["tier"][0]][1]
contrib["alpha_kelly"] = contrib.index.map(alpha_cont).fillna(1.0)
contrib["alpha_tier"] = contrib.index.map(alpha_tier_cont).fillna(1.0)
contrib = contrib.reset_index()
contrib["test_roi"] = contrib["test_pnl"] / contrib["total_pnl"].replace(0, np.nan)
contrib.to_csv(OUT_DIR / "wallet_scaling_contrib.csv", index=False)

a = contrib["alpha_kelly"].to_numpy()
ts = contrib["test_sharpe"].to_numpy()
valid = np.isfinite(ts)
rho = spearman_rho(pd.Series(a[valid]), pd.Series(ts[valid])) if valid.sum() > 2 else np.nan
print(f"Spearman(alpha_kelly, wallet test sharpe) = {rho:.4f}  (n={int(valid.sum())})")
contrib[["wallet", "alpha_kelly", "alpha_tier", "test_pnl", "test_sharpe", "test_roi"]].head(15)


Spearman(alpha_kelly, wallet test sharpe) = 0.1886  (n=29)


,wallet,alpha_kelly,alpha_tier,test_pnl,test_sharpe,test_roi
0,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0.6459,0.0000,-3270.7787,-0.6723,-4.6178
1,0x0e3226217752d7247c67b870b72b99ba3e20535b,0.7506,0.0000,-662.9597,-1.1228,-0.6083
2,0x0fe90c22827e72c7aef24bdc52b6392943fd4fd9,1.5599,2.0000,87.1176,7.7649,0.0200
3,0x10eba4e0c9857d3e5421295ae5962abe9af23c1f,0.4954,0.0000,26.8516,11.0303,0.1570
4,0x183a2a0b034877e761af0778da5134bebc37d514,0.8029,2.0000,21885.4085,5.4778,0.5822
5,0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,1.4374,1.0000,-1928.7526,-4.6179,-0.4534
6,0x2bcd792138a4e184f791b05d9c5c70e3c8f7cbfb,0.6720,0.0000,548.0812,0.6256,0.6800
7,0x37a46125cfa561a8b52c4a6ddd093a65bb4ca487,0.6864,1.0000,50.6808,9.1425,0.0598
8,0x5042e6dc8a612c493881a3e67519cc09f5f4fcb0,0.4739,0.0000,123.4105,2.0522,1.2964
9,0x551e72eda42a5ab39d6d78239a1d9bbb5db6b0e0,0.2065,2.0000,-423.2102,-7.0171,-0.0038


## Save stage 1 result

In [28]:
import json
from datetime import datetime, timezone

best_name = max(best_per_scheme.values(), key=lambda x: x[2])[0]
best_params = schemes[best_name][2]

wallet_cols = [
    "wallet", "mu", "sigma", "n_days", "total_pnl", "sharpe_proxy",
    "alpha_kelly", "alpha_tier", "test_pnl", "test_n_days", "test_sharpe", "test_roi",
]
wallet_records = contrib[[c for c in wallet_cols if c in contrib.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

test_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == best_name)].iloc[0]
copy_all_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == "copy_all")].iloc[0]

metadata = {
    "type": "scaled_copy",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": int((contrib["alpha_tier"] > 0).sum()),
    "n_wallets_total": len(wallets),
    "split_sizes": {k: int(len(v)) for k, v in splits.items()},
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_val_sharpe": float(max(best_per_scheme.values(), key=lambda x: x[2])[2]),
    "test_performance": {
        "config": best_name,
        "trades": int(test_row["trades"]),
        "pnl": float(test_row["pnl"]),
        "roi_w": float(test_row["roi_w"]),
        "sharpe_daily": float(test_row["sharpe_daily"]),
        "copy_all": {
            "trades": int(copy_all_row["trades"]),
            "pnl": float(copy_all_row["pnl"]),
            "roi_w": float(copy_all_row["roi_w"]),
            "sharpe_daily": float(copy_all_row["sharpe_daily"]),
        },
    },
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = NB_DIR / "stage1_scaled_result.json"
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 scaled result -> {out_path.resolve()}")


Saved stage 1 scaled result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_scaled_result.json


## Price-scaling fill experiment (exploratory)

Test a limit-price entry idea on a **sample** (~1k test contracts, copy-default wallets):
copy each candidate copy-wallet BUY at `limit = price * scale` for
`scale ∈ {1.0, 0.98, 0.95, 0.90}` and give the order a **5-minute window** to fill.

- **Fill rule:** filled iff within `(dt, dt+5min]` any trade on the same
  `(condition_id, token_id)` prints at `price <= limit` with a strictly greater timestamp.
- **Fill price:** exactly the limit price, so
  `pnl = copyable_pnl + copyable_qty * (price - limit)` (same formula/quantity as the
  original `copyable_pnl`); unfilled trades contribute 0.
- **Baseline:** `scale = 1.0` is the market-copy (fill immediately at `price`), so it
  must reproduce `sum(copyable_pnl)` on the sample.


In [29]:
from lib import DEFAULT_TRADES_DIR
from signal_lab.wallet_scaling import price_scale_fill_sim

rng = np.random.RandomState(42)
test_markets = np.sort(splits["test"]["condition_id"].unique())
n_sel = min(1000, len(test_markets))
sel_markets = rng.choice(test_markets, size=n_sel, replace=False)
signals = splits["test"][splits["test"]["condition_id"].isin(sel_markets)].copy()
signals = signals[signals["copyable_qty"] > 0]
print(f"test markets: {len(test_markets):,}  sampled: {n_sel:,}")
print(f"candidate BUYs (copyable_qty>0) on sample: {len(signals):,}")

_tape_cols = ["condition_id", "token_id", "dt", "avg_price"]
tape_parts = []
for f in sorted(DEFAULT_TRADES_DIR.glob("*.parquet")):
    tp = pd.read_parquet(f, columns=_tape_cols)
    tp = tp[tp["condition_id"].isin(sel_markets)]
    if not tp.empty:
        tape_parts.append(tp.rename(columns={"avg_price": "price"}))
tape = (
    pd.concat(tape_parts, ignore_index=True)
    if tape_parts
    else pd.DataFrame(columns=["condition_id", "token_id", "dt", "price"])
)
print(f"fill tape rows (sampled contracts, both sides): {len(tape):,}")


test markets: 1,613  sampled: 1,000
candidate BUYs (copyable_qty>0) on sample: 3,575
fill tape rows (sampled contracts, both sides): 2,638,784


In [30]:
SCALES = (1.0, 0.98, 0.95, 0.90)
sim = price_scale_fill_sim(signals, tape, scales=SCALES, window_minutes=5.0)
base_pnl = float(signals["copyable_pnl"].sum())

summary = (
    sim.groupby("scale")
    .agg(signals=("filled", "size"), fills=("filled", "sum"),
         fill_rate=("filled", "mean"), pnl=("pnl", "sum"))
    .reset_index()
)
summary["pnl_pct_of_market"] = summary["pnl"] / base_pnl * 100 if base_pnl else np.nan
summary["delta_vs_market"] = summary["pnl"] - base_pnl
print(f"market-copy pnl (baseline = sum copyable_pnl): {base_pnl:,.2f}")
summary.round(2)


market-copy pnl (baseline = sum copyable_pnl): 57,686.96


,scale,signals,fills,fill_rate,pnl,pnl_pct_of_market,delta_vs_market
0,0.9000,3575,576,0.1600,17058.9400,29.5700,-40628.0200
1,0.9500,3575,942,0.2600,32728.2900,56.7300,-24958.6700
2,0.9800,3575,1236,0.3500,48677.4300,84.3800,-9009.5400
3,1.0000,3575,3575,1.0000,57686.9600,100.0000,0.0000


In [31]:
pw_pnl = sim.pivot_table(index="wallet", columns="scale", values="pnl", aggfunc="sum")
pw_fill = sim.pivot_table(index="wallet", columns="scale", values="filled", aggfunc="mean")
pw = pw_pnl.join(pw_fill.rename(columns={c: f"fill_{c:g}" for c in pw_fill.columns}))
pw = pw.reindex(pw[1.0].sort_values(ascending=False).index)
pw.round(1).head(15)


scale,0.9000,0.9500,0.9800,1.0000,fill_0.9,fill_0.95,fill_0.98,fill_1
wallet,,,,,,,,
0x57c5de7efafb020e589f80e0da6e16e9b5c907aa,12602.1000,23492.1000,31407.2000,32779.0000,0.0000,0.4000,0.5000,1.0000
0x183a2a0b034877e761af0778da5134bebc37d514,3465.4000,7415.5000,10238.0000,11533.9000,0.1000,0.1000,0.2000,1.0000
0xe1b361d6a6f237b9ed7534d19b232df8369e1426,1011.2000,2363.7000,6010.2000,8433.3000,0.1000,0.2000,0.3000,1.0000
0x8b72cb885a6bd4ea9d1da393ca231f0fa3476dbe,978.2000,812.0000,2103.2000,3133.1000,0.3000,0.4000,0.5000,1.0000
0xe471565da86af7b6e99d9c955b0b35d1f0e0c87c,2962.8000,2523.7000,2688.0000,2716.5000,0.2000,0.2000,0.3000,1.0000
0xe27757cc1dff108d82cb2ce371a40ef7a52b8e46,-16.9000,207.5000,219.5000,2623.5000,0.1000,0.2000,0.5000,1.0000
0xf72044fb3f98854411a069d9a682e54e4b824599,0.0000,526.1000,508.8000,2587.5000,0.0000,0.2000,0.2000,1.0000
0x2bcd792138a4e184f791b05d9c5c70e3c8f7cbfb,719.7000,843.1000,902.4000,874.6000,0.3000,0.4000,0.5000,1.0000
0x742ff6df87485287bb4db5b0d7fa7af13047d673,8.7000,101.2000,143.9000,478.6000,0.0000,0.2000,0.3000,1.0000


In [32]:
sim.to_csv(OUT_DIR / "price_scale_sim.csv", index=False)
summary.round(4).to_csv(OUT_DIR / "price_scale_summary.csv", index=False)
pw.round(2).reset_index().to_csv(OUT_DIR / "price_scale_wallets.csv", index=False)
print("saved -> signal_lab/price_scale_{sim,summary,wallets}.csv")


saved -> signal_lab/price_scale_{sim,summary,wallets}.csv


## Cumulative PnL by variant

Total pnl per simulation config, summed across all splits, shown as a bar chart.

In [40]:
import plotly.graph_objects as go

cum_pnl = (
    sim_df.groupby(["scheme", "config"], as_index=False)["pnl"].sum()
    .sort_values("pnl", ascending=False)
)
print("Cumulative pnl by variant (sum over splits):")
for _, r in cum_pnl.iterrows():
    print(f"  {r['scheme']:10s}  {r['config']:>22s}  pnl={r['pnl']:>14,.2f}")

fig = go.Figure()
for key, (name, _params, _val_sharpe) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, COST_SEL)
    daily = res["daily_pnl"].sort_index()
    cum = daily.cumsum()
    fig.add_trace(
        go.Scatter(
            x=cum.index,
            y=cum.values,
            mode="lines",
            name=name,
            hovertemplate="%{x|%Y-%m-%d}<br>pnl=%{y:,.2f}<extra>%{fullData.name}</extra>",
        )
    )
fig.update_layout(
    title="Cumulative PnL over time (test split)",
    xaxis_title="date",
    yaxis_title="cumulative pnl",
    height=500,
    legend_title="config",
)
fig.show()


Cumulative pnl by variant (sum over splits):
  tier                     tier3@2-0  pnl=     61,753.16
  uniform                uniform@0.5  pnl=     30,351.33
  tier                     tier3@4-0  pnl=     29,786.98
  tier                     tier3@8-0  pnl=     29,786.98
  tier                  tier3@8-0.25  pnl=     28,367.92
  tier                  tier5@8-0.25  pnl=     28,175.96
  tier                  tier3@4-0.25  pnl=     27,798.33
  tier                  tier5@2-0.25  pnl=     27,551.08
  tier                     tier5@4-0  pnl=     27,389.62
  tier                     tier5@8-0  pnl=     27,389.62
  tier                     tier5@2-0  pnl=     27,389.62
  tier                  tier5@4-0.25  pnl=     27,232.82
  tier                     tier4@8-0  pnl=     26,766.18
  tier                     tier4@2-0  pnl=     26,766.18
  tier                     tier4@4-0  pnl=     26,766.18
  tier                  tier4@4-0.25  pnl=     26,108.48
  tier                  tier4@8-0.25  pnl= 